# 09. Defenses and ranking metrics (RQ5)

Produces, from the saved results:

- **Table VII** (`reports/table7_defenses_by_dataset.csv`, mean over datasets in `table7_defenses.csv`): defended
  worst-case ASR and clean AUC per dataset, for the tabular models (ivis with its adversarial augmentation).
- **Table S-XXIX** (`reports/tables/defenses_context.csv`): the same for the graph and foundation models, constrained
  (C) beside consistent (K).
- **Figure S6** (`fig_defense_tradeoff`): what each defense buys and costs.
- **Figure S7** (`fig_pareto`): clean-robust Pareto frontiers, with the LoRA / zero-shot foundation models when
  `FM_ADAPT_RUNS` points at their runs.
- **Tables S-XXX and S-XXXI** (`reports/tables/robust_auc_mean.csv`, `reports/table9_robust_auc.csv`): AUC and KS
  under a score-minimising attacker.

Inputs: `runs/<d>/defenses/results*.csv` (`defenses/run_defenses.py`; the later runs `results_<family>_<defense>.csv`
from `defenses/run_defenses_consistent.py --both`, whose consistent half is
`runs_consistent/<d>/defenses/results_consistent_<family>_<defense>.csv`), the attack grid `runs/<d>/attacks/results*.csv`
and `per_example*.npz`, `runs/<d>/attacks/robust_auc*.csv` (`attacks/robust_auc.py`), the models' `metrics.json`, and
optionally the LoRA / zero-shot runs in `$FM_ADAPT_RUNS` (`study-artifacts/fm_adapt/runs_v2`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

The summary functions write into `reports/`; create the folders as `scripts/make_results.sh` does, and load the
attack grid (every `runs/<d>/attacks/results*.csv` in one frame).

In [ ]:
import os
from fraudadv.report import make_report as MR
from fraudadv.report import tables

(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)
A = MR.load_attacks()
print(len(A), "rows of the attack grid")

## LoRA and zero-shot foundation models (points of Figure S7)

`fraudadv.report.fm_adapt` summarises the LoRA-adapted and zero-shot Chronos-2 / TimesFM runs (Table S-XXVIII,
`reports/table17_fm_adapt.csv`) and writes their Pareto points to `reports/pareto_extra_fm_adapt.csv`, which
`pareto_figure` reads. It runs first, as in `scripts/make_results.sh`. Skipped when `FM_ADAPT_RUNS` is not set.

In [ ]:
from fraudadv.paths import RUNS
from fraudadv.report import fm_adapt

FM_ADAPT_RUNS = os.environ.get("FM_ADAPT_RUNS")
fm_files = []
if FM_ADAPT_RUNS:
    fm_adapt.main(RUNS, fm_adapt.RL_ROOT / "results_adapt", ROOT / "reports", Path(FM_ADAPT_RUNS))
    fm_files = ["reports/table17_fm_adapt.csv", "reports/pareto_extra_fm_adapt.csv"]
else:
    print("FM_ADAPT_RUNS is not set: Figure S7 is drawn without the LoRA / zero-shot points (it will differ from the study's)")

## Inputs of the Pareto figure: clean performance and worst-case ASR

`fig_pareto` reads `reports/table1_clean.csv` (`make_report.table_clean`) and `reports/table6_worstcase_ci.csv`
(`make_report.stats_tables`, which also writes the other statistics of notebook 10).

In [ ]:
MR.table_clean()
MR.stats_tables(A)
nb.show_csv("reports/table6_worstcase_ci.csv", rows=5);

## Table VII and Figure S6

`make_report.defense_tables` keeps, per dataset, model and defense, the worst case of the adaptive attacks at
eps = 0.25 in the dataset's defense threat model (constrained; unconstrained on ULB), writes
`reports/table7_defenses_by_dataset.csv` (Table VII), the mean over datasets (`table7_defenses.csv`) and Figure S6
(`fig_defense_tradeoff`, with its data in `table7_defense_tradeoff.csv`).

In [ ]:
MR.defense_tables(A)
nb.show_csv("reports/table7_defenses.csv", rows=10);

In [ ]:
nb.show_csv("reports/table7_defenses_by_dataset.csv", rows=12);

## Table S-XXIX: defenses of the graph and foundation models

`consistent_tables.defenses_context` takes the HGNN, HGT, Chronos-2 and TimesFM rows of
`reports/table7_defenses_by_dataset.csv` (the C column and clean AUC, written by `defense_tables` above) and the consistent
attacker's rows of `runs_consistent/<d>/defenses/results_consistent*.csv` (the K column, on IEEE-CIS, Sparkov and PaySim),
into `reports/tables/defenses_context.csv`. These defenses were run at eps = 0.25 only, each defended model trained once
and attacked by both attackers (`run_defenses_consistent --both`). The foundation models are attacked as in the grid, on
240 random frauds of the sample.

In [ ]:
from fraudadv.report import consistent_tables as CT

CT.defenses_context()
nb.show_csv("reports/tables/defenses_context.csv", rows=12);

## Figure S7

`make_report.pareto_figure` draws the clean-robust Pareto frontier of every model and defense per dataset
(`fraudadv.report.pareto`): clean recall at 1% FPR against robust recall = recall x (1 - worst-case ASR). Its points are in
`reports/table16_pareto.csv`.

In [ ]:
MR.pareto_figure()
assert not (ROOT / "reports/fig_pareto.err").exists(), (ROOT / "reports/fig_pareto.err").read_text()
nb.show_csv("reports/table16_pareto.csv", rows=8);

In [ ]:
nb.show_figure("fig_defense_tradeoff")

In [ ]:
nb.show_figure("fig_pareto")

## Tables S-XXX and S-XXXI

`make_report.robust_auc_table` sets the change in AUC and KS under the score-minimising attacker
(`runs/<d>/attacks/robust_auc*.csv`) beside the min-effort attacker of the grid, at eps = 0.25
(`reports/table9_robust_auc.csv`, Table S-XXXI); `tables.robust_auc_mean` writes the constrained mean over datasets
(`reports/tables/robust_auc_mean.csv`, Table S-XXX).

In [ ]:
MR.robust_auc_table(A)
tables.robust_auc_mean()
nb.show_csv("reports/table9_robust_auc.csv", rows=8);

In [ ]:
nb.show_csv("reports/tables/robust_auc_mean.csv");

## Run it yourself: the squeezing defense on ULB XGBoost

Rebuilds the feature-squeezing detector around the saved ULB XGBoost (`run_defenses.train_defended`, defense
`"squeeze"`: delta and the score threshold calibrated on validation to flag 1% of legitimate traffic) and attacks the whole
system with Square at eps = 0.25 (`run_defenses.evaluate`, which returns rows and writes nothing). `n` only applies when
the grid's fraud sample is missing; here the grid's ULB sample (52 frauds) is used, so this is the study's own cell, not
a smaller illustration. The result is compared with the stored row of `runs/ulb/defenses/results.csv`. About 5 seconds.

In [ ]:
import pandas as pd
from fraudadv.attacks.constraints import Constraints
from fraudadv.data.load import load
from fraudadv.defenses import run_defenses as RD

a, meta = load("ulb")
mode = RD.DEFENSE_MODE.get("ulb", "constrained")                  # unconstrained on ULB
cons = Constraints(meta, a["X_train"], mode)
squeezed = RD.train_defended("ulb", "xgboost", "squeeze", a, meta, cons)
live = pd.DataFrame(RD.evaluate("ulb", "xgboost", "squeeze", squeezed, a, cons, n=50, mode=mode, eps=[0.25]))

stored = pd.read_csv(ROOT / "runs/ulb/defenses/results.csv")
stored = stored[(stored.model == "xgboost") & (stored.defense == "squeeze") & (stored.eps == 0.25)]
cols = ["attack", "n_detected", "asr", "asr_usd", "clean_auc", "clean_recall@1%fpr"]
display(live[cols].assign(source="live"), stored[cols].assign(source="stored"))
print("live equals stored:", all(abs(live[c].values - stored[c].values).max() < 1e-9 for c in cols[1:]))

## Rerun the runs behind these results from scratch

The calls `scripts/reproduce.sh` makes for this step, for one dataset: the score-minimising attacker
(`attacks/robust_auc.py`), the defenses with their adaptive attacks (`defenses/run_defenses.py` for the tabular models;
`defenses/run_defenses_consistent.py --both` for ivis's augmentation and the graph and foundation models) and, on IEEE-CIS,
Sparkov and PaySim, the defenses under the consistent attacker (`defenses/run_defenses_consistent.py`, Table S-XX). They
take hours (days for every dataset) and write into `runs/<dataset>/` and `runs_consistent/<dataset>/`; they skip work
whose outputs exist. Not run here.

In [ ]:
RERUN = False
DATASET = "ulb"
if RERUN:
    import os
    import subprocess

    def py(*args):
        """python3 -m <module> <args> from the folder root, as py() in scripts/reproduce.sh."""
        subprocess.run([sys.executable, "-m", *args], check=True, cwd=ROOT,
                       env=dict(os.environ, PYTHONPATH=str(ROOT / "src")))

    # what runs on this dataset (as in scripts/reproduce.sh)
    TAB = "xgboost,catboost,mlp,ft_transformer,ivis"
    CTX = "" if DATASET in ("ulb", "baf") else "hgnn,hgt"
    FMS = "" if DATASET in ("ulb", "baf") else "chronos2,timesfm"
    ALL = TAB + ("," + CTX if CTX else "")
    CONSISTENT = DATASET in ("ieee_cis", "sparkov", "paysim")

    py("fraudadv.attacks.robust_auc", DATASET, ALL, "--tag", "_all")
    if FMS:
        py("fraudadv.attacks.robust_auc", DATASET, FMS, "--tag", "_fm")
    TAB_DEFENSES = "none,pgd_at,trades,adv_boost,monotone,squeeze"
    py("fraudadv.defenses.run_defenses", DATASET, "--defenses", TAB_DEFENSES)
    # the defenses added later, at eps 0.25: each model trained once, attacked by C and (K datasets) by K
    py("fraudadv.defenses.run_defenses_consistent", DATASET, "--families", "ivis", "--defenses", "adv_aug",
       "--eps", "0.25", "--tag", "_ivis_adv_aug", "--both")
    for f in [x for x in (CTX + "," + FMS).split(",") if x]:
        for x in ("none", "pgd_at", "trades", "squeeze"):
            py("fraudadv.defenses.run_defenses_consistent", DATASET, "--families", f, "--defenses", x,
               "--eps", "0.25", "--tag", f"_{f}_{x}", "--both")
    if CONSISTENT:
        py("fraudadv.defenses.run_defenses_consistent", DATASET, "--defenses", TAB_DEFENSES)

## Check against the reference build

Every file this notebook wrote, compared byte for byte with the same path under `$FRAUDADV_REFERENCE`.

In [ ]:
nb.check(fm_files + [
    "reports/table1_clean.csv", "reports/table1_clean.md",
    "reports/table6_worstcase_ci.csv", "reports/table6_friedman.csv", "reports/table6_mcnemar.csv",
    "reports/table6_mixed_model.csv", "reports/table6_mixed_model_nopaysim.csv",
    "reports/table7_defenses_by_dataset.csv", "reports/table7_defense_tradeoff.csv",
    "reports/table7_defenses.csv", "reports/table7_defenses.md",
    "reports/fig_defense_tradeoff.pdf", "reports/fig_defense_tradeoff.png",
    "reports/table16_pareto.csv", "reports/fig_pareto.pdf", "reports/fig_pareto.png",
    "reports/table9_robust_auc.csv", "reports/table9_robust_auc.md",
    "reports/tables/defenses_context.csv", "reports/tables/robust_auc_mean.csv",
])